> ### **Target Clean**

In [0]:
# Define paths for the Medallion Architecture
BRONZE_PATH = "/Volumes/workspace/accenture_final_project/lakehouse/bronze/"
SILVER_PATH = "/Volumes/workspace/accenture_final_project/lakehouse/silver/"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import pyspark.sql.functions as F

In [0]:
# Define the schema to enforce data types upon reading (Schema Enforcement)
target_schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Target", StringType(), True),
    StructField("TargetMonth", StringType(), True)
])

In [0]:
# Read data from Bronze layer
df_target_raw = (spark.read
                 .format("csv")
                 .schema(target_schema)
                 .option("header", "true")
                 .option("sep", "\t")
                 .option("ignoreLeadingWhiteSpace", "true")
                 .option("ignoreTrailingWhiteSpace", "true")
                 .option("quote", '"')
                 .option("escape", '"')
                 .load(BRONZE_PATH + "Targets.csv")
                )

In [0]:
display(df_target_raw)

In [0]:
# Count initial rows for Data Quality metrics (Useful for the presentation)
initial_count = df_target_raw.count()

In [0]:
# Silver Layer Transformations (Data Cleansing)
df_target_silver = (df_target_raw
    # Clean Currency: Remove '$' and ',' symbols, then cast to Double
    .withColumn("Target", F.regexp_replace(F.col("Target"), r'[\$,]', '').cast(DoubleType()))

    # Clean Date (e.g., 'Friday, December 1, 2017' -> 2017-12-01)
    # Extract only the "Month Day, Year" part to avoid parsing issues
    .withColumn("CleanDate", F.substring_index(F.col("TargetMonth"), ", ", -2))
    .withColumn("TargetMonth", F.to_date(F.col("CleanDate"), "MMMM d, yyyy"))
    .drop("CleanDate") # Drop temporary column

    # Remove duplicate records
    .dropDuplicates()
)

In [0]:
display(df_target_silver)

In [0]:
# --- OUTLIER & ANOMALY HANDLING ---
# Keep only valid records: Target must be positive (no negative targets) and not null
df_target_clean = df_target_silver.filter(
    (F.col("Target").isNotNull()) & 
    (F.col("Target") > 0) &
    (F.col("TargetMonth").isNotNull())
)

In [0]:
# Calculate metrics for the presentation
final_count = df_target_clean.count()
print(f"Data Quality Metrics for Targets:")
print(f"Initial rows (Bronze): {initial_count}")
print(f"Final valid rows (Silver): {final_count}")
print(f"Rows removed (Duplicates/Outliers/Nulls): {initial_count - final_count}")
print(f"Data Retention Rate: {(final_count / initial_count) * 100:.2f}%")

## **Transform to DELTA**

In [0]:
# Write to Silver Layer as Delta format
(df_target_clean.write
 .mode("overwrite")
 .format("delta")
 .save(SILVER_PATH + "targets_clean"))